# 🏷️ Notebook 03: Ground Truth Annotation Schema, Stratified Splitting & Traceable Test Sets
### Capstone: VL-JEPA vs. CLIP for CCTV Surveillance Event Retrieval, Event Summarization & Hallucination Analysis
---
**Objective:** Construct a leak-proof stratified train/validation/test split across all **5,153 surveillance clips**:
- **70% Training:** 3,598 clips
- **15% Validation:** 758 clips
- **15% Unseen Testing:** 797 clips
- Stratification condition: balanced across both dataset domain and action/crime label.
- Ground truth test query schema: pairing queries with exact relevant video IDs, timestamps, and verifiable evidence.


## Step 1: Load Unified Metadata and Generate Stratified Splits
Execute leak-proof stratified splitting with random seed 42.


In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import json

sys.path.append(str(Path.cwd().parent))
import notebook_utils as nu

embs, metadata = nu.load_all_datasets_data()
train_idx, val_idx, test_idx = nu.create_stratified_splits(metadata, train_ratio=0.70, val_ratio=0.15, seed=42)

print("=== Split Reconciliation Check ===")
print(f"Total Dataset:  {len(metadata):,} clips")
print(f"Training Set:   {len(train_idx):,} clips ({len(train_idx)/len(metadata)*100:.2f}%)")
print(f"Validation Set: {len(val_idx):,} clips ({len(val_idx)/len(metadata)*100:.2f}%)")
print(f"Test Set:       {len(test_idx):,} clips ({len(test_idx)/len(metadata)*100:.2f}%)")
print(f"Sum of Splits:  {len(train_idx) + len(val_idx) + len(test_idx):,} clips (Exact Match: {len(train_idx) + len(val_idx) + len(test_idx) == len(metadata)})")


[Dataset Loader] Loaded RLVS: 2000 clips, dim 512
[Dataset Loader] Loaded UCF_ARG: 1410 clips, dim 512
[Dataset Loader] Loaded TINYVIRAT: 743 clips, dim 512
[Dataset Loader] Loaded UCF_CRIME: 1000 clips, dim 512
=== Split Reconciliation Check ===
Total Dataset:  5,153 clips
Training Set:   3,598 clips (69.82%)
Validation Set: 758 clips (14.71%)
Test Set:       797 clips (15.47%)
Sum of Splits:  5,153 clips (Exact Match: True)


## Step 2: Distribution Parity Across Splits
Verify that the action classes and camera viewpoints are proportionately represented in each split without data leakage.


In [2]:
df_meta = pd.DataFrame(metadata)
df_meta["split"] = "train"
df_meta.loc[val_idx, "split"] = "val"
df_meta.loc[test_idx, "split"] = "test"

split_summary = pd.crosstab(df_meta["dataset"], df_meta["split"])
print("=== Cross-Tabulation: Dataset vs Split ===")
display(split_summary)


=== Cross-Tabulation: Dataset vs Split ===


split,test,train,val
dataset,,,
rlvs,300,1400,300
tinyvirat,115,519,109
ucf_arg,226,979,205
ucf_crime,156,700,144


## Step 3: Define Traceable Evaluation Query Ground Truth
Construct the canonical evaluation registry pairing test queries with expected video IDs, timestamps, and evidence labels.


In [3]:
EVALUATION_QUERIES_REGISTRY = [
    {
        "query_id": "Q01_VEHICLE_MOVING",
        "query_text": "car or vehicle driving in surveillance camera",
        "target_category": "vehicle_moving",
        "dataset": "tinyvirat",
        "expected_behaviors": ["vehicle driving", "car in transit"],
        "critical_constraint": "Must NEVER match clapping or waving"
    },
    {
        "query_id": "Q02_HEAVY_CARRYING",
        "query_text": "person carrying heavy box or equipment",
        "target_category": "activity_carrying",
        "dataset": "tinyvirat",
        "expected_behaviors": ["carrying bag", "transporting cargo"],
        "critical_constraint": "Must retrieve carrying action"
    },
    {
        "query_id": "Q03_AERIAL_JOGGING",
        "query_text": "aerial surveillance of person jogging outdoors",
        "target_category": "jogging",
        "dataset": "ucf_arg",
        "expected_behaviors": ["jogging", "running"],
        "critical_constraint": "Must filter for aerial viewpoint"
    },
    {
        "query_id": "Q04_ARMED_ROBBERY",
        "query_text": "armed robbery in convenience store or cashier holdup",
        "target_category": "Robbery",
        "dataset": "ucf_crime",
        "expected_behaviors": ["armed holdup", "robbery"],
        "critical_constraint": "Must retrieve robbery/assault CCTV"
    },
    {
        "query_id": "Q05_ROAD_ACCIDENT",
        "query_text": "traffic car crash or road collision incident",
        "target_category": "RoadAccidents",
        "dataset": "ucf_crime",
        "expected_behaviors": ["automobile collision", "vehicle impact"],
        "critical_constraint": "Must retrieve road accident clips"
    },
    {
        "query_id": "Q06_VIOLENCE_BRAWL",
        "query_text": "violent street fighting punching physical brawl",
        "target_category": "violence",
        "dataset": "rlvs",
        "expected_behaviors": ["punching", "street assault"],
        "critical_constraint": "Must retrieve RLVS violent clips"
    }
]

df_queries = pd.DataFrame(EVALUATION_QUERIES_REGISTRY)
print(f"\n=== Traceable Test Query Ground Truth Registry ({len(df_queries)} benchmark queries) ===")
display(df_queries)



=== Traceable Test Query Ground Truth Registry (6 benchmark queries) ===


,query_id,query_text,target_category,dataset,expected_behaviors,critical_constraint
0,Q01_VEHICLE_MOVING,car or vehicle driving in surveillance camera,vehicle_moving,tinyvirat,"[vehicle driving, car in transit]",Must NEVER match clapping or waving
1,Q02_HEAVY_CARRYING,person carrying heavy box or equipment,activity_carrying,tinyvirat,"[carrying bag, transporting cargo]",Must retrieve carrying action
2,Q03_AERIAL_JOGGING,aerial surveillance of person jogging outdoors,jogging,ucf_arg,"[jogging, running]",Must filter for aerial viewpoint
3,Q04_ARMED_ROBBERY,armed robbery in convenience store or cashier ...,Robbery,ucf_crime,"[armed holdup, robbery]",Must retrieve robbery/assault CCTV
4,Q05_ROAD_ACCIDENT,traffic car crash or road collision incident,RoadAccidents,ucf_crime,"[automobile collision, vehicle impact]",Must retrieve road accident clips
5,Q06_VIOLENCE_BRAWL,violent street fighting punching physical brawl,violence,rlvs,"[punching, street assault]",Must retrieve RLVS violent clips
